In [ ]:
# Installs the scikit-learn machine learning library into the Google Colab environment
!pip install scikit-learn

In [ ]:
# Import the necessary Colab tools for authentication and accessing Google Drive
from google.colab import auth, drive

# Authenticate for Google services (including Sheets)
auth.authenticate_user()

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import gspread
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from google.auth import default
# New import for calculating performance metrics
from sklearn.metrics import accuracy_score, recall_score, f1_score

# --- 1. CONFIGURATION: UPDATE THESE THREE VARIABLES ---
# Define the paths to your labeled image folders and their corresponding labels.
# Use label 1 for 'soft story' and 0 for 'non-soft story'.
FOLDER_CONFIG = {
    # Path to folder with soft story images
    '/content/drive/MyDrive/CEE-247C/247C Project/data/soft_story_images/soft_story_images/images': 1, # <-- CHANGE ME!
    # Path to folder with NON-soft story images
    '/content/drive/MyDrive/CEE-247C/247C Project/data/non_soft_story_images/images': 0, # <-- CHANGE ME!
}

# The name for your results spreadsheet.
# This will be created in the root of your Google Drive if it doesn't exist.
spreadsheet_name = 'Soft_Story_Validation_Results_Improved_Final'
# --- END OF CONFIGURATION ---


# --- 2. SETUP GOOGLE SHEETS ---
print("Setting up Google Sheets...")
try:
    creds, _ = default()
    gc = gspread.authorize(creds)
    # Try to open the spreadsheet by name, create it if it doesn't exist.
    try:
        sh = gc.open(spreadsheet_name)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(spreadsheet_name)
        print(f"Created new spreadsheet: '{spreadsheet_name}'")

    worksheet = sh.sheet1
    # Clear the sheet to avoid appending to old data
    worksheet.clear()
    print(f"Successfully connected to spreadsheet '{spreadsheet_name}'.")
except Exception as e:
    print(f"Error connecting to Google Sheets: {e}")
    worksheet = None # Set to None so the script can proceed without it

# --- 3. LOAD THE AI MODEL ---
print("Loading the Vision-Language Model (CLIP)...")
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)
print("Model loaded successfully.")

# --- 4. DEFINE CLASSIFICATION LABELS ---
labels = [
    "a photo of a soft story building with large openings or windows on the ground floor",
    "a photo of a standard building without a soft story"
]

# --- 5. PROCESS IMAGES AND EVALUATE ---
print("\nStarting image processing and evaluation...")

# Lists to store results for metric calculation
all_actuals = []
all_predictions = []
# List to hold all our data for bulk-uploading to Google Sheets
sheet_data = [['Building ID', 'Predicted', 'Actual']]

# Loop through each configured folder (e.g., soft-story folder, then non-soft-story folder)
for folder_path, actual_label in FOLDER_CONFIG.items():
    print(f"\nProcessing folder: {folder_path} (Actual Label: {actual_label})")

    # Get a list of all image files in the current folder
    try:
        image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        if not image_files:
            print("  -> Warning: No image files found in this folder.")
            continue # Move to the next folder
    except FileNotFoundError:
        print(f"  -> Error: The folder '{folder_path}' was not found. Please check your path.")
        continue # Move to the next folder

    total_images = len(image_files)
    for i, filename in enumerate(image_files):
        original_path = os.path.join(folder_path, filename)
        print(f"  Processing {i+1}/{total_images}: {filename}")

        try:
            image = Image.open(original_path)

            # Prepare inputs for the model
            inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

            # Get model prediction
            with torch.no_grad():
                outputs = model(**inputs)

            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
            # Probability of the first label ("soft story")
            soft_story_prob = probs[0][0].item()

            # --- Determine predicted label ---
            # If probability of 'soft story' is > 0.5, predict 1, else predict 0
            predicted_label = 1 if soft_story_prob > 0.5 else 0

            # Append results for later metric calculation
            all_actuals.append(actual_label)
            all_predictions.append(predicted_label)

            # Extract building ID from filename (assuming it's the name without extension)
            building_id, _ = os.path.splitext(filename)

            # Add the result to our list for Google Sheets
            sheet_data.append([building_id, predicted_label, actual_label])

        except Exception as e:
            print(f"    -> Could not process file {filename}. Error: {e}")
            sheet_data.append([filename, 'Error processing', actual_label])

# --- 6. UPLOAD RESULTS TO GOOGLE SHEETS ---
if worksheet and sheet_data:
    print("\nUploading results to Google Sheets...")
    try:
        # Update the worksheet with all the data at once
        worksheet.update('A1', sheet_data)
        print("Upload complete.")
    except Exception as e:
        print(f"  -> Error uploading to Google Sheets: {e}")

# --- 7. CALCULATE AND DISPLAY PERFORMANCE METRICS ---
if not all_actuals or not all_predictions:
    print("\nNo images were processed, cannot calculate metrics.")
else:
    print("\n--- Performance Metrics ---")
    # Note: `pos_label=1` assumes '1' is the positive class (soft story)
    # `zero_division=0` prevents errors if a metric is undefined (e.g., recall when no actual positives exist)
    accuracy = accuracy_score(all_actuals, all_predictions)
    recall = recall_score(all_actuals, all_predictions, pos_label=1, zero_division=0)
    f1 = f1_score(all_actuals, all_predictions, pos_label=1, zero_division=0)

    print(f"Total Images Processed: {len(all_actuals)}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})")
    print(f"Recall (Sensitivity): {recall:.4f} ({recall:.2%})")
    print(f"F1 Score: {f1:.4f} ({f1:.2%})")
    print("---------------------------\n")

Setting up Google Sheets...
Created new spreadsheet: 'Soft_Story_Validation_Results_Improved_Final'
Successfully connected to spreadsheet 'Soft_Story_Validation_Results_Improved_Final'.
Loading the Vision-Language Model (CLIP)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


流式输出内容被截断，只能显示最后 5000 行内容。
  Processing 2351/7348: 2974_view6.jpg
  Processing 2352/7348: 2975_view0.jpg
  Processing 2353/7348: 2976_view0.jpg
  Processing 2354/7348: 2976_view2.jpg
  Processing 2355/7348: 2977_view0.jpg
  Processing 2356/7348: 2977_view2.jpg
  Processing 2357/7348: 2978_view0.jpg
  Processing 2358/7348: 2978_view2.jpg
  Processing 2359/7348: 2978_view3.jpg
  Processing 2360/7348: 2979_view0.jpg
  Processing 2361/7348: 2979_view2.jpg
  Processing 2362/7348: 2980_view0.jpg
  Processing 2363/7348: 2981_view0.jpg
  Processing 2364/7348: 2982_view2.jpg
  Processing 2365/7348: 2983_view0.jpg
  Processing 2366/7348: 2983_view1.jpg
  Processing 2367/7348: 2984_view0.jpg
  Processing 2368/7348: 2984_view5.jpg
  Processing 2369/7348: 2985_view0.jpg
  Processing 2370/7348: 2985_view4.jpg
  Processing 2371/7348: 2986_view0.jpg
  Processing 2372/7348: 2986_view3.jpg
  Processing 2373/7348: 2987_view0.jpg
  Processing 2374/7348: 2987_view2.jpg
  Processing 2375/7348: 2987_view3.jp

/tmp/ipykernel_348/534552739.py:125: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update('A1', sheet_data)


Upload complete.

--- Performance Metrics ---
Total Images Processed: 18028
Accuracy: 0.4536 (45.36%)
Recall (Sensitivity): 0.1772 (17.72%)
F1 Score: 0.2776 (27.76%)
---------------------------

